![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

# **Clinical Entity Resolvers: From Clinical Text to Standard Codes**

**Healthcare NLP Certification Training, Fall 2026**

Clinical notes describe the same concept in many ways: *"type 2 DM"*, *"T2DM"*, *"type two diabetes mellitus"*, *"adult-onset diabetes"*. NER finds these mentions, but billing, analytics, registries and AI agents need **one standard code** for each concept.

**Entity resolution** maps every extracted entity to a code in a standard medical terminology such as **ICD-10-CM**, **RxNorm**, **SNOMED CT** or **LOINC**.

### **In this notebook**

1. What entity resolution is and why it matters
2. How Sentence Entity Resolvers work
3. Resolving single terms: RxNorm and SNOMED CT
4. End-to-end pipeline: clinical note → ICD-10-CM codes (with HCC)
5. Resolving only what the patient has: `AssertionFilterer`
6. Several terminologies in one pipeline: `Router`, plus structured JSON output
7. Optional: giving the resolver more than the chunk: `BertSentenceChunkEmbeddings` and `EntityChunkEmbeddings`
8. The shortcut: pretrained resolver pipelines
9. Takeaways and resources

## **Setup**

This notebook downloads several GB of models. A **High-RAM** Colab runtime is recommended (*Runtime → Change runtime type*).

In [ ]:
import json
import os

from google.colab import files

if 'spark_jsl.json' not in os.listdir():
  license_keys = files.upload()
  os.rename(list(license_keys.keys())[0], 'spark_jsl.json')

with open('spark_jsl.json') as f:
    license_keys = json.load(f)

# Defining license key-value pairs as local variables
locals().update(license_keys)
os.environ.update(license_keys)

In [ ]:
# Installing pyspark and spark-nlp
! pip install --upgrade -q pyspark==3.5.1 spark-nlp==$PUBLIC_VERSION

# Installing Spark NLP Healthcare
! pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION  --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Installing Spark NLP Display Library for visualization
! pip install -q spark-nlp-display

In [3]:
import json
import os

import sparknlp
import sparknlp_jsl

from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.pretrained import PretrainedPipeline
from sparknlp_jsl.pipeline_tracer import PipelineTracer
from sparknlp_jsl.pipeline_output_parser import PipelineOutputParser

from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel

import pandas as pd
pd.set_option('display.max_colwidth', 200)

import warnings
warnings.filterwarnings('ignore')

params = {"spark.driver.memory":"16G",
          "spark.kryoserializer.buffer.max":"2000M",
          "spark.driver.maxResultSize":"2000M"}

spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)

print("Spark NLP Version :", sparknlp.version())
print("Spark NLP_JSL Version :", sparknlp_jsl.version())

spark

Spark NLP Version : 7.0.0
Spark NLP_JSL Version : 7.0.0


## **1. What is Entity Resolution?**

NER answers **"what is mentioned, and where?"**. Entity resolution answers **"which standard concept is this?"**

| Mention in the note | Terminology | Code | Standard concept |
|---|---|---|---|
| gestational diabetes mellitus | ICD-10-CM | O24.4 | Gestational diabetes mellitus |
| chest pain on left side | SNOMED CT | 285385002 | Left sided chest pain |
| Coumadin | RxNorm | 202421 | Coumadin |

**Why it matters**

- **Medical coding and billing:** ICD-10-CM diagnoses, CPT procedures, HCC risk adjustment
- **Interoperability:** EHRs, FHIR resources and registries exchange coded data, not free text
- **Analytics and cohort building:** "all patients with type 2 diabetes" becomes a single code query
- **AI agents and RAG:** coded entities give LLMs structured, unambiguous facts to work with

**Common terminologies**

| Terminology | What it covers | Example resolver model |
|---|---|---|
| ICD-10-CM | Diagnoses (US) | [`sbiobertresolve_icd10cm_augmented_billable_hcc`](https://nlp.johnsnowlabs.com/2026/07/14/sbiobertresolve_icd10cm_augmented_billable_hcc_en.html) |
| ICD-10-PCS | Inpatient procedures (US) | [`sbiobertresolve_icd10pcs_augmented`](https://nlp.johnsnowlabs.com/2026/07/15/sbiobertresolve_icd10pcs_augmented_en.html) |
| CPT | Procedures and services* | [`sbiobertresolve_cpt_procedures_measurements_augmented`](https://nlp.johnsnowlabs.com/2026/07/16/sbiobertresolve_cpt_procedures_measurements_augmented_en.html) |
| RxNorm | Clinical drugs | [`sbiobertresolve_rxnorm_augmented`](https://nlp.johnsnowlabs.com/2024/06/18/sbiobertresolve_rxnorm_augmented_en.html) |
| SNOMED CT | Findings, disorders, procedures, body structures | [`sbiobertresolve_snomed_findings_20260901`](https://nlp.johnsnowlabs.com/2026/09/11/sbiobertresolve_snomed_findings_20260901_en.html)† |
| LOINC | Lab tests and observations | [`sbiobertresolve_loinc_augmented_2_83`](https://nlp.johnsnowlabs.com/2026/09/06/sbiobertresolve_loinc_augmented_2_83_en.html)† |
| UMLS | Concept IDs (CUIs) across 100+ vocabularies | [`sbiobertresolve_umls_major_concepts`](https://nlp.johnsnowlabs.com/2026/06/11/sbiobertresolve_umls_major_concepts_en.html) |
| HPO | Human phenotypes | [`sbiobertresolve_HPO`](https://nlp.johnsnowlabs.com/2026/07/20/sbiobertresolve_HPO_en.html) |

\* CPT models need a separate CPT license and can't be downloaded with `.pretrained()`.

† Needs Healthcare NLP 6.4.1 or later.

All resolver models: [Models Hub → Entity Resolution](https://nlp.johnsnowlabs.com/models?task=Entity+Resolution&edition=Spark+NLP+for+Healthcare)

## **2. How Sentence Entity Resolvers Work**

A Sentence Entity Resolver is a **semantic nearest-neighbour search** over a terminology:

1. **When the model is trained:** every code description and its synonyms are embedded with a sentence-embeddings model and stored inside the resolver.
2. **At runtime:** each NER chunk is embedded with the **same** sentence-embeddings model.
3. The resolver finds the closest terminology entries and returns the best code plus the **top-k candidates**.

The embeddings come from **SBERT-style (Siamese) models** such as `sbiobert_base_cased_mli`, which are trained to place phrases with the same meaning close together. That's why *"type two diabetes"* and *"T2DM"* land near the same code even though they share almost no words.

> ⚠️ **Use the embeddings model the resolver was trained with.** It's listed on each model card (in this notebook, the Section 3 models use `sbiobert_base_cased_mli`, and the 2026 models in Sections 4–6 use its ONNX version, `sbiobert_base_cased_mli_onnx`). A mismatched embeddings model returns wrong codes without raising any error.

### **The resolver pipeline**

```
DocumentAssembler            text                      → document
SentenceDetectorDLModel      document                  → sentence
Tokenizer                    sentence                  → token
WordEmbeddingsModel          sentence, token           → word_embeddings
MedicalNerModel              sentence, token, word_emb → ner
NerConverterInternal         sentence, token, ner      → ner_chunk        (keep only labels you want to code)
Chunk2Doc                    ner_chunk                 → doc_ner_chunk
BertSentenceEmbeddings       doc_ner_chunk             → sentence_embeddings
SentenceEntityResolverModel  sentence_embeddings       → icd10cm_code
```

The first six stages are a regular clinical NER pipeline. Resolution adds only three: `Chunk2Doc`, `BertSentenceEmbeddings` and `SentenceEntityResolverModel`.

### **What the resolver returns**

The annotation's `result` is the best code. Its `metadata` holds the details:

| Field | Meaning |
|---|---|
| `target_text` | The chunk that was resolved |
| `resolved_text` | Best-matching terminology entry |
| `distance` | Distance between the chunk and the top match |
| `all_k_results` | Top-k candidate codes, separated by `:::` |
| `all_k_resolutions` | Descriptions of the top-k candidates |
| `all_k_distances`, `all_k_cosine_distances` | Distances of the top-k candidates |
| `all_k_aux_labels` | Extra information for some models, e.g. billable / HCC flags for ICD-10-CM |

### **Helper function**

Turns the resolver output of a `LightPipeline` result into a pandas table. Resolver metadata already contains the chunk text and the NER label, so this works for every pipeline in the notebook.

In [4]:
def get_codes(light_result, resolver_col, top_k=5, hcc=False):
    """Returns the resolver annotations of one LightPipeline.fullAnnotate() result as a DataFrame.

    light_result : one element of LightPipeline.fullAnnotate(...)
    resolver_col : output column of the SentenceEntityResolverModel
    top_k        : number of candidate codes to show
    hcc          : split ICD-10-CM aux labels into billable / HCC columns (top match)
    """
    rows = []
    for annotation in light_result[resolver_col]:
        meta = annotation.metadata
        row = {
            "chunk": meta.get("target_text"),
            "entity": meta.get("entity"),
            "code": annotation.result,
            "resolved_text": meta.get("resolved_text"),
            "top_k_codes": meta.get("all_k_results", "").split(":::")[:top_k],
            "top_k_resolutions": meta.get("all_k_resolutions", "").split(":::")[:top_k],
        }
        if hcc:
            # aux label format of the ICD-10-CM HCC models: billable||hcc_status||hcc_score
            top_aux = meta.get("all_k_aux_labels", "").split(":::")[0]
            billable, hcc_status, hcc_score = (top_aux.split("||") + ["", "", ""])[:3]
            row.update({"billable": billable, "hcc_status": hcc_status, "hcc_score": hcc_score})
        rows.append(row)

    return pd.DataFrame(rows).dropna(axis=1, how="all")

## **3. Resolving Single Terms**

Sometimes the terms are already isolated: a medication list, a problem-list column, a form field. Then NER isn't needed. The text goes straight into the embeddings and the resolver:

`DocumentAssembler → BertSentenceEmbeddings → SentenceEntityResolverModel`

This is also the quickest way to see what a resolver does.

### **RxNorm**

RxNorm, maintained by the US National Library of Medicine, assigns a code to each clinical drug: **active ingredient + strength + dose form**. Brand names and generics get their own codes and are linked to each other.

In [5]:
documentAssembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("ner_chunk")

sbert_embedder = BertSentenceEmbeddings.pretrained("sbiobert_base_cased_mli", "en", "clinical/models")\
    .setInputCols(["ner_chunk"])\
    .setOutputCol("sentence_embeddings")\
    .setCaseSensitive(False)

rxnorm_resolver = SentenceEntityResolverModel.pretrained("sbiobertresolve_rxnorm_augmented", "en", "clinical/models")\
    .setInputCols(["sentence_embeddings"])\
    .setOutputCol("rxnorm_code")\
    .setDistanceFunction("EUCLIDEAN")

rxnorm_pipeline = PipelineModel(stages=[
    documentAssembler,
    sbert_embedder,
    rxnorm_resolver
])

rxnorm_lp = LightPipeline(rxnorm_pipeline)

sbiobert_base_cased_mli download started this may take some time.
Approximate size to download 384.3 MB
[OK!]
sbiobertresolve_rxnorm_augmented download started this may take some time.
Approximate size to download 1.1 GB
[OK!]


`LightPipeline` runs the pipeline on plain Python strings, without building a Spark DataFrame. It's ideal for quick checks and small inputs.

In [6]:
drug_terms = ["metformin 100 mg", "Coumadin", "amoxicillin 500 mg oral capsule"]

pd.concat([get_codes(result, "rxnorm_code") for result in rxnorm_lp.fullAnnotate(drug_terms)], ignore_index=True)

,chunk,code,resolved_text,top_k_codes,top_k_resolutions
0,metformin 100 mg,861024,metformin hydrochloride 100 mg/ml [metformin hydrochloride 100 mg/ml],"[861024, 334738, 332848, 861026, 333262]","[metformin hydrochloride 100 mg/ml [metformin hydrochloride 100 mg/ml], fenofibrate 100 mg [fenofibrate 100 mg], ciprofibrate 100 mg [ciprofibrate 100 mg], metformin hydrochloride 100 mg/ml [riome..."
1,Coumadin,202421,coumadin [coumadin],"[202421, 2898, 1855075, 128793, 1598]","[coumadin [coumadin], coumarin [coumarin], coumaran [coumaran], vicodin [vicodin], dicoumarol [dicumarol]]"
2,amoxicillin 500 mg oral capsule,308191,amoxicillin 500 mg oral capsule [amoxicillin 500 mg oral capsule],"[308191, 200998, 308192, 200999, 200997]","[amoxicillin 500 mg oral capsule [amoxicillin 500 mg oral capsule], amoxicillin 500 mg oral capsule [amoxil] [amoxicillin 500 mg oral capsule [amoxil]], amoxicillin 500 mg oral tablet [amoxicillin..."


- `resolved_text` is the matched terminology entry, followed by the official description in brackets.
- `top_k_codes` are the nearest alternatives. When the top match is wrong or too generic, the right code is often among them, which is what a human reviewer needs.

### **SNOMED CT: swap the resolver, keep everything else**

SNOMED CT is the most comprehensive clinical terminology, covering findings, disorders, procedures and body structures. To resolve to SNOMED CT instead of RxNorm, keep the same embeddings and **change only the resolver model**.

In [7]:
snomed_resolver = SentenceEntityResolverModel.pretrained("sbiobertresolve_snomed_findings", "en", "clinical/models")\
    .setInputCols(["sentence_embeddings"])\
    .setOutputCol("snomed_code")\
    .setDistanceFunction("EUCLIDEAN")

snomed_pipeline = PipelineModel(stages=[
    documentAssembler,
    sbert_embedder,
    snomed_resolver
])

snomed_lp = LightPipeline(snomed_pipeline)

finding_terms = ["brain tumor", "chest pain on left side", "short of breath"]

pd.concat([get_codes(result, "snomed_code") for result in snomed_lp.fullAnnotate(finding_terms)], ignore_index=True)

sbiobertresolve_snomed_findings download started this may take some time.
Approximate size to download 174.5 MB
[OK!]


,chunk,code,resolved_text,top_k_codes,top_k_resolutions
0,brain tumor,422840005,mass lesion of brain,"[422840005, 396922003, 396925001, 396923008, 396924002]","[mass lesion of brain, who grade i (central nervous system tumor), who grade iv (central nervous system tumor), who grade ii (central nervous system tumor), who grade iii (central nervous system t..."
1,chest pain on left side,285385002,left sided chest pain,"[285385002, 427293006, 15743561000119103, 287047008, 316801000119101]","[left sided chest pain, pain radiating to left side of chest, chronic pain of left upper limb, pain in left leg, pain of left lower leg]"
2,short of breath,267036007,shortness of breath,"[267036007, 60845006, 86684002, 83156004, 276572003]","[shortness of breath, short of breath on exertion, slow breathing, diminished sense of smell, poor respiratory drive]"


*"chest pain on left side"* matches *"left sided chest pain"*: different words, same meaning, same code. That's the semantic search at work.

## **4. End-to-End: Clinical Note → ICD-10-CM**

In real notes the entities aren't isolated, so NER finds the problems first and the resolver codes each one.

**ICD-10-CM** is the US diagnosis code set: about 70,000 codes of 3 to 7 characters (e.g. `E11.9`). The model used here, `sbiobertresolve_icd10cm_augmented_billable_hcc`, also returns for each candidate:

- **billable:** whether the code is specific enough to be used on a claim
- **HCC status / HCC score:** whether the code maps to a Hierarchical Condition Category, which CMS uses for risk adjustment

> **Match NER labels to the terminology.** ICD-10-CM codes diagnoses, so we keep only `PROBLEM` chunks with `setWhiteList`. A drug or a lab test sent to an ICD-10-CM resolver still gets a code, just a wrong one.

In [8]:
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

sentence_detector = SentenceDetectorDLModel.pretrained("sentence_detector_dl_healthcare", "en", "clinical/models")\
    .setInputCols(["document"])\
    .setOutputCol("sentence")

tokenizer = Tokenizer()\
    .setInputCols(["sentence"])\
    .setOutputCol("token")

word_embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
    .setInputCols(["sentence", "token"])\
    .setOutputCol("word_embeddings")

clinical_ner = MedicalNerModel.pretrained("ner_clinical", "en", "clinical/models")\
    .setInputCols(["sentence", "token", "word_embeddings"])\
    .setOutputCol("ner")

ner_converter = NerConverterInternal()\
    .setInputCols(["sentence", "token", "ner"])\
    .setOutputCol("ner_chunk")\
    .setWhiteList(["PROBLEM"])

chunk2doc = Chunk2Doc()\
    .setInputCols(["ner_chunk"])\
    .setOutputCol("doc_ner_chunk")

# the 2026 ICD-10-CM models are trained with the ONNX version of sbiobert (see the model card)
chunk_embeddings = BertSentenceEmbeddings.pretrained("sbiobert_base_cased_mli_onnx", "en", "clinical/models")\
    .setInputCols(["doc_ner_chunk"])\
    .setOutputCol("sentence_embeddings")\
    .setCaseSensitive(False)

icd10_resolver = SentenceEntityResolverModel.pretrained("sbiobertresolve_icd10cm_augmented_billable_hcc", "en", "clinical/models")\
    .setInputCols(["sentence_embeddings"])\
    .setOutputCol("icd10cm_code")\
    .setDistanceFunction("EUCLIDEAN")

icd10_pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer,
    word_embeddings,
    clinical_ner,
    ner_converter,
    chunk2doc,
    chunk_embeddings,
    icd10_resolver
])

empty_data = spark.createDataFrame([[""]]).toDF("text")

icd10_model = icd10_pipeline.fit(empty_data)

sentence_detector_dl_healthcare download started this may take some time.
Approximate size to download 367.3 KB
[OK!]
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
ner_clinical download started this may take some time.
Approximate size to download 13.9 MB
[OK!]
sbiobert_base_cased_mli_onnx download started this may take some time.
Approximate size to download 384.4 MB
[OK!]
sbiobertresolve_icd10cm_augmented_billable_hcc download started this may take some time.
Approximate size to download 1.2 GB
[OK!]


### **Run at scale with a Spark DataFrame**

In production there are thousands of notes in a table. `transform()` distributes the work across the cluster.

In [9]:
clinical_note = (
    'A 28-year-old female with a history of gestational diabetes mellitus diagnosed eight years '
    'prior to presentation and subsequent type two diabetes mellitus, associated '
    'with an acute hepatitis, and obesity with a body mass index (BMI) of 33.5 kg/m2, '
    'presented with a one-week history of polyuria, polydipsia, poor appetite, and vomiting. '
    'Two weeks prior to presentation, she was treated with a five-day course of amoxicillin '
    'for a respiratory tract infection. She had been on dapagliflozin for six months '
    'at the time of presentation. Physical examination on presentation was significant for dry oral mucosa; '
    'significantly, her abdominal examination was benign with no tenderness or guarding. Pertinent '
    'laboratory findings on admission were: serum glucose 111 mg/dl, bicarbonate 18 mmol/l, anion gap 20, '
    'creatinine 0.4 mg/dL, triglycerides 508 mg/dL, total cholesterol 122 mg/dL, glycated hemoglobin (HbA1c) '
    '10%, and venous pH 7.27. Serum lipase was normal at 43 U/L. Serum acetone levels could not be assessed '
    'as blood samples kept hemolyzing due to significant lipemia. The patient was initially admitted for '
    'starvation ketosis, as she reported poor oral intake for three days prior to admission.')

notes_df = spark.createDataFrame([[clinical_note]]).toDF("text")

icd10_result = icd10_model.transform(notes_df)

In [10]:
icd10_result.select(F.explode("icd10cm_code").alias("res"))\
    .select(F.col("res.metadata")["target_text"].alias("chunk"),
            F.col("res.result").alias("icd10cm_code"),
            F.col("res.metadata")["resolved_text"].alias("resolved_text"))\
    .show(truncate=False)

+-------------------------------------+------------+-------------------------------------------------------------------------------------------+
|chunk                                |icd10cm_code|resolved_text                                                                              |
+-------------------------------------+------------+-------------------------------------------------------------------------------------------+
|gestational diabetes mellitus        |O24.4       |gestational diabetes mellitus [gestational diabetes mellitus]                              |
|subsequent type two diabetes mellitus|O24.11      |pre-existing type 2 diabetes mellitus [pre-existing type 2 diabetes mellitus, in pregnancy]|
|an acute hepatitis                   |B15         |acute hepatitis a [acute hepatitis a]                                                      |
|obesity                              |E66.9       |obesity [obesity, unspecified]                                                

### **Inspect the results with LightPipeline**

For a single note, `LightPipeline` is faster and returns Python objects we can pass to our helper. `hcc=True` adds the billable and HCC columns for the top match.

In [11]:
icd10_lp = LightPipeline(icd10_model)

icd10_light_result = icd10_lp.fullAnnotate(clinical_note)[0]

get_codes(icd10_light_result, "icd10cm_code", hcc=True)

,chunk,entity,code,resolved_text,top_k_codes,top_k_resolutions,billable,hcc_status,hcc_score
0,gestational diabetes mellitus,PROBLEM,O24.4,gestational diabetes mellitus [gestational diabetes mellitus],"[O24.4, O24.41, O24.43, Z86.32, Z87.5]","[gestational diabetes mellitus [gestational diabetes mellitus], gestational diabetes mellitus (disorder) [gestational diabetes mellitus in pregnancy], postpartum gestational diabetes mellitus [ges...",0,0,0
1,subsequent type two diabetes mellitus,PROBLEM,O24.11,"pre-existing type 2 diabetes mellitus [pre-existing type 2 diabetes mellitus, in pregnancy]","[O24.11, E11.8, E11, E13.9, E11.9]","[pre-existing type 2 diabetes mellitus [pre-existing type 2 diabetes mellitus, in pregnancy], disorder associated with type 2 diabetes mellitus [type 2 diabetes mellitus with unspecified complicat...",0,0,0
2,an acute hepatitis,PROBLEM,B15,acute hepatitis a [acute hepatitis a],"[B15, K72.0, B17.9, B17.2, Z03.89]","[acute hepatitis a [acute hepatitis a], acute hepatitis [acute and subacute hepatic failure], acute infectious hepatitis [acute viral hepatitis, unspecified], acute hepatitis e [acute hepatitis e]...",0,0,0
3,obesity,PROBLEM,E66.9,"obesity [obesity, unspecified]","[E66.9, E66.8, Z68.41, Q13.0, E66]","[obesity [obesity, unspecified], abdominal obesity [other obesity], obese [body mass index [bmi] 40.0-44.9, adult], central obesity [coloboma of iris], overweight and obesity [overweight and obesi...",1,0,0
4,a body mass index,PROBLEM,Z68.41,"finding of body mass index [body mass index [bmi] 40.0-44.9, adult]","[Z68.41, E66.9, R22.9, Z68.1, R22.3]","[finding of body mass index [body mass index [bmi] 40.0-44.9, adult], observation of body mass index [obesity, unspecified], mass of body region [localized swelling, mass and lump, unspecified], f...",1,1,48
5,polyuria,PROBLEM,R35,polyuria [polyuria],"[R35, R35.81, E23.2, R35.89, R31]","[polyuria [polyuria], nocturnal polyuria [nocturnal polyuria], polyuric state (disorder) [diabetes insipidus], other polyuria [other polyuria], hematuria [hematuria]]",0,0,0
6,polydipsia,PROBLEM,R63.1,polydipsia [polydipsia],"[R63.1, F63.89, E23.2, F63.9, O40]","[polydipsia [polydipsia], psychogenic polydipsia [other impulse disorders], primary polydipsia [diabetes insipidus], psychogenic polydipsia (disorder) [impulse disorder, unspecified], polyhydramni...",1,0,0
7,poor appetite,PROBLEM,R63.0,poor appetite [anorexia],"[R63.0, P92.9, R43.8, R43.2, E86]","[poor appetite [anorexia], poor feeding [feeding problem of newborn, unspecified], bad taste in mouth [other disturbances of smell and taste], unpleasant taste in mouth [parageusia], poor fluid in...",1,0,0
8,vomiting,PROBLEM,R11.1,vomiting [vomiting],"[R11.1, R11, R11.10, G43.A1, P92.1]","[vomiting [vomiting], intermittent vomiting [nausea and vomiting], vomiting symptoms [vomiting, unspecified], periodic vomiting [cyclical vomiting, in migraine, intractable], finding of vomiting [...",0,0,0
9,a respiratory tract infection,PROBLEM,J98.8,respiratory tract infection [other specified respiratory disorders],"[J98.8, J06.9, A49.9, J22, J20.9]","[respiratory tract infection [other specified respiratory disorders], upper respiratory tract infection [acute upper respiratory infection, unspecified], bacterial respiratory infection [bacterial...",1,0,0


**What to look for**

- Most problems resolve to an exact match, e.g. *obesity*, *polyuria* and *vomiting*.
- Check the `billable` column. When the top code is a category rather than a billable code (`billable = 0`; for example *O24.4*, gestational diabetes mellitus, which has more specific child codes), the specific codes are usually in `top_k_codes`. That's why coders need the alternatives.
- **The resolver only sees the chunk.** *"a body mass index"* carries no value, so the resolver can't know the BMI is 33.5. *"subsequent type two diabetes mellitus"* carries extra words that can pull the code away from plain type 2 diabetes. Better chunks give better codes.

### **Visualize**

`sparknlp_display` highlights each chunk in the original text together with its code and description.

In [12]:
from sparknlp_display import EntityResolverVisualizer

visualiser = EntityResolverVisualizer()
visualiser.set_label_colors({"PROBLEM": "#008080"})

visualiser.display(icd10_light_result, "ner_chunk", "icd10cm_code")

## **5. Resolve Only What the Patient Has: `AssertionFilterer`**

Not every problem in a note belongs to the patient right now: *"**no** stomach pain"*, *"**father** with Alzheimer"*, *"short of breath **with** climbing stairs"*. Coding these would put wrong diagnoses on a claim or a patient record.

Two stages go between NER and the resolver:

- `AssertionDLModel` labels each chunk: `present`, `absent`, `hypothetical`, `conditional`, `possible`, `associated_with_someone_else`
- `AssertionFilterer` keeps only the chunks whose assertion is in the white list (here: `present`)

All other stages are reused from the ICD-10-CM pipeline above.

In [13]:
clinical_assertion = AssertionDLModel.pretrained("assertion_dl", "en", "clinical/models")\
    .setInputCols(["sentence", "ner_chunk", "word_embeddings"])\
    .setOutputCol("assertion")

assertion_filterer = AssertionFilterer()\
    .setInputCols(["sentence", "ner_chunk", "assertion"])\
    .setOutputCol("assertion_filtered")\
    .setCaseSensitive(False)\
    .setWhiteList(["present"])

present_chunk2doc = Chunk2Doc()\
    .setInputCols(["assertion_filtered"])\
    .setOutputCol("doc_ner_chunk")

assertion_pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer,
    word_embeddings,
    clinical_ner,
    ner_converter,
    clinical_assertion,
    assertion_filterer,
    present_chunk2doc,
    chunk_embeddings,
    icd10_resolver
])

assertion_lp = LightPipeline(assertion_pipeline.fit(empty_data))

assertion_dl download started this may take some time.
Approximate size to download 1.3 MB
[OK!]


In [14]:
assertion_note = """Patient with severe fever and sore throat. He shows no stomach pain and he maintained on an epidural and PCA for pain control.
He also became short of breath with climbing a flight of stairs. After CT, lung tumor located at the right lower lobe. Father with Alzheimer."""

assertion_result = assertion_lp.fullAnnotate(assertion_note)[0]

# every PROBLEM chunk with its assertion status
pd.DataFrame({
    "chunk": [chunk.result for chunk in assertion_result["ner_chunk"]],
    "assertion": [assertion.result for assertion in assertion_result["assertion"]],
})

,chunk,assertion
0,severe fever,present
1,sore throat,present
2,stomach pain,absent
3,pain control,hypothetical
4,short of breath,conditional
5,lung tumor,present
6,Alzheimer,associated_with_someone_else


In [15]:
# only the "present" chunks reach the resolver
get_codes(assertion_result, "icd10cm_code")

,chunk,entity,code,resolved_text,top_k_codes,top_k_resolutions
0,severe fever,PROBLEM,R50.9,"sustained fever [fever, unspecified]","[R50.9, P81.9, R50.8, A23.0, A92.8]","[sustained fever [fever, unspecified], continuous fever [disturbance of temperature regulation of newborn, unspecified], chronic fever [other specified fever], undulant fever [brucellosis due to b..."
1,sore throat,PROBLEM,R07.0,sore throat [pain in throat],"[R07.0, J02.9, J31.2, R09.89, K13.7]","[sore throat [pain in throat], sore throat present [acute pharyngitis, unspecified], persistent sore throat [chronic pharyngitis], throat tightness [other specified symptoms and signs involving th..."
2,lung tumor,PROBLEM,D49.1,tumor of lung [neoplasm of unspecified behavior of respiratory system],"[D49.1, C34.90, D38.1, C34.9, C78.00]","[tumor of lung [neoplasm of unspecified behavior of respiratory system], malignant tumor of lung [malignant neoplasm of unspecified part of unspecified bronchus or lung], tumor of lung parenchyma ..."


Absent, hypothetical, conditional and family-history problems are dropped **before** resolution, so only the patient's current problems get codes.

## **6. Several Terminologies in One Pipeline: `Router`**

A note usually contains problems **and** drugs, so we want ICD-10-CM for the problems and RxNorm for the drugs.

Without `Router` you'd run `BertSentenceEmbeddings` once per resolver. With `Router`, embeddings are computed **once** for all chunks, and each chunk's embedding is sent to the right resolver based on its NER label:

```
ner_chunk (PROBLEM + TREATMENT) → Chunk2Doc → BertSentenceEmbeddings
                                                 ├── Router(PROBLEM)   → problem_embeddings   → ICD-10-CM resolver
                                                 └── Router(TREATMENT) → treatment_embeddings → RxNorm resolver
```

> `ner_clinical` labels drugs as `TREATMENT`. In production, a dedicated drug NER such as `ner_posology` (label `DRUG`) gives cleaner drug chunks, including strength and form.

In [16]:
router_ner_converter = NerConverterInternal()\
    .setInputCols(["sentence", "token", "ner"])\
    .setOutputCol("ner_chunk")\
    .setWhiteList(["PROBLEM", "TREATMENT"])

problem_router = Router()\
    .setInputCols(["sentence_embeddings"])\
    .setFilterFieldsElements(["PROBLEM"])\
    .setOutputCol("problem_embeddings")

treatment_router = Router()\
    .setInputCols(["sentence_embeddings"])\
    .setFilterFieldsElements(["TREATMENT"])\
    .setOutputCol("treatment_embeddings")

# slim: a lighter ICD-10-CM model that also returns billable / HCC information
router_icd10_resolver = SentenceEntityResolverModel.pretrained("sbiobertresolve_icd10cm_slim_billable_hcc", "en", "clinical/models")\
    .setInputCols(["problem_embeddings"])\
    .setOutputCol("icd10cm_code")\
    .setDistanceFunction("EUCLIDEAN")

# v2: RxNorm model trained with the same ONNX embeddings as the ICD-10-CM model
router_rxnorm_resolver = SentenceEntityResolverModel.pretrained("sbiobertresolve_rxnorm_augmented_v2", "en", "clinical/models")\
    .setInputCols(["treatment_embeddings"])\
    .setOutputCol("rxnorm_code")\
    .setDistanceFunction("EUCLIDEAN")

router_pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer,
    word_embeddings,
    clinical_ner,
    router_ner_converter,
    chunk2doc,
    chunk_embeddings,
    problem_router,
    treatment_router,
    router_icd10_resolver,
    router_rxnorm_resolver
])

router_model = router_pipeline.fit(empty_data)
router_lp = LightPipeline(router_model)

sbiobertresolve_icd10cm_slim_billable_hcc download started this may take some time.
Approximate size to download 423.7 MB
[OK!]
sbiobertresolve_rxnorm_augmented_v2 download started this may take some time.
Approximate size to download 1.3 GB
[OK!]


In [17]:
router_note = """The patient is a 41-year-old Vietnamese female with a cough that started last week.
She has had right-sided chest pain radiating to her back with fever starting yesterday.
She has a history of pericarditis in May 2006 and developed cough with right-sided chest pain.
MEDICATIONS
1. Coumadin 1 mg daily. Last INR was on Tuesday, August 14, 2007, and her INR was 2.3.
2. Amiodarone 100 mg p.o. daily.
"""

router_result = router_lp.fullAnnotate(router_note)[0]

get_codes(router_result, "icd10cm_code", hcc=True)

,chunk,entity,code,resolved_text,top_k_codes,top_k_resolutions,billable,hcc_status,hcc_score
0,a cough,PROBLEM,R05,cough [cough],"[R05, R05.3, R05.1, A37, R05.2]","[cough [cough], chronic cough [chronic cough], acute cough [acute cough], whooping cough [whooping cough], subacute cough [subacute cough]]",0,0,0
1,right-sided chest pain,PROBLEM,R10.11,right upper quadrant pain [right upper quadrant pain],"[R10.11, M79.621, M79.604, M79.601, M25.511]","[right upper quadrant pain [right upper quadrant pain], pain in right upper arm [pain in right upper arm], pain in right leg [pain in right leg], pain in right arm [pain in right arm], pain in rig...",1,0,0
2,fever,PROBLEM,A68,relapsing fevers [relapsing fevers],"[A68, A78, R50.2, M04.1, A96.2]","[relapsing fevers [relapsing fevers], q fever [q fever], drug induced fever [drug induced fever], periodic fever syndromes [periodic fever syndromes], lassa fever [lassa fever]]",0,0,0
3,pericarditis,PROBLEM,I30.1,infective pericarditis [infective pericarditis],"[I30.1, B33.23, I30, I31.0, I01.0]","[infective pericarditis [infective pericarditis], viral pericarditis [viral pericarditis], acute pericarditis [acute pericarditis], chronic adhesive pericarditis [chronic adhesive pericarditis], a...",1,0,0
4,cough,PROBLEM,R05,cough [cough],"[R05, A37, R05.3, R05.1, R05.2]","[cough [cough], whooping cough [whooping cough], chronic cough [chronic cough], acute cough [acute cough], subacute cough [subacute cough]]",0,0,0
5,right-sided chest pain,PROBLEM,R10.11,right upper quadrant pain [right upper quadrant pain],"[R10.11, M79.621, M79.604, M79.601, M25.511]","[right upper quadrant pain [right upper quadrant pain], pain in right upper arm [pain in right upper arm], pain in right leg [pain in right leg], pain in right arm [pain in right arm], pain in rig...",1,0,0


In [18]:
get_codes(router_result, "rxnorm_code")

,chunk,entity,code,resolved_text,top_k_codes,top_k_resolutions
0,Coumadin,TREATMENT,202421,coumadin [coumadin],"[202421, 2898, 1855075, 2239, 128793]","[coumadin [coumadin], coumarin [coumarin], coumaran [coumaran], cephradin [cephradine], vicodin [vicodin]]"
1,Amiodarone,TREATMENT,703,amiodarone [amiodarone],"[703, 1663223, 1151983, 203114, 1663270]","[amiodarone [amiodarone], amiodarone injection [amiodarone injection], amiodarone pill [amiodarone pill], amiodarone hcl [amiodarone hydrochloride], amiodarone injection [nexterone] [amiodarone in..."


One embeddings pass, two terminologies.

Notice that NER caught *"Coumadin"* and *"Amiodarone"* without the dose or the route (*1 mg*, *100 mg p.o.*). The resolver can only use what's in the chunk, so check the resolved drug form against the note. Again, **chunk quality = code quality**.

### **From annotations to structured JSON**

Downstream systems (databases, FHIR, LLM agents) need clean, structured output. `PipelineTracer` inspects the pipeline and builds a parser configuration; `PipelineOutputParser` turns the LightPipeline output into JSON with entities and resolutions linked by `chunk_id`.

In [19]:
column_maps = PipelineTracer(router_model).createParserDictionary()
column_maps.update({"document_identifier": "router_demo"})

parsed = PipelineOutputParser(column_maps).run([router_result])

resolutions = [{key: value for key, value in resolution.items() if "confidence" not in key}
               for resolution in parsed["result"][0]["resolutions"][:2]]

print(json.dumps(resolutions, indent=2))

[
  {
    "vocab": "icd10cm_code",
    "chunk_id": "2361c21e",
    "chunk": "a cough",
    "code": "R05",
    "resolutions": "cough [cough]",
    "all_k_codes": "R05:::R05.3:::R05.1:::A37:::R05.2:::R06.01:::R06.7:::R06.0:::R06.2:::R06:::J38.4:::R05.4:::G44.83:::R00.2:::R06.00:::R07.1:::R07.0:::R09.81:::R12:::S22.5:::R49.0:::R07:::L29:::R06.83:::G47.51",
    "all_k_resolutions": "cough [cough]:::chronic cough [chronic cough]:::acute cough [acute cough]:::whooping cough [whooping cough]:::subacute cough [subacute cough]:::orthopnea [orthopnea]:::sneezing [sneezing]:::dyspnea [dyspnea]:::wheezing [wheezing]:::abnormalities of breathing [abnormalities of breathing]:::edema of larynx [edema of larynx]:::cough syncope [cough syncope]:::primary cough headache [primary cough headache]:::palpitations [palpitations]:::dyspnea, unspecified [dyspnea, unspecified]:::chest pain on breathing [chest pain on breathing]:::pain in throat [pain in throat]:::nasal congestion [nasal congestion]:::heartburn 

In [20]:
entities_df = pd.DataFrame(parsed["result"][0]["entities"])
resolutions_df = pd.DataFrame(parsed["result"][0]["resolutions"])

entities_df.merge(resolutions_df, on=["chunk_id", "chunk"])[["chunk", "ner_label", "vocab", "code", "resolutions"]]

,chunk,ner_label,vocab,code,resolutions
0,a cough,PROBLEM,icd10cm_code,R05,cough [cough]
1,right-sided chest pain,PROBLEM,icd10cm_code,R10.11,right upper quadrant pain [right upper quadrant pain]
2,fever,PROBLEM,icd10cm_code,A68,relapsing fevers [relapsing fevers]
3,pericarditis,PROBLEM,icd10cm_code,I30.1,infective pericarditis [infective pericarditis]
4,cough,PROBLEM,icd10cm_code,R05,cough [cough]
5,right-sided chest pain,PROBLEM,icd10cm_code,R10.11,right upper quadrant pain [right upper quadrant pain]
6,Coumadin,TREATMENT,rxnorm_code,202421,coumadin [coumadin]
7,Amiodarone,TREATMENT,rxnorm_code,703,amiodarone [amiodarone]


## **7. Optional: Give the Resolver More Than the Chunk**

Sections 4 and 6 showed the main limitation: **the resolver only sees the chunk**. Two annotators can replace `Chunk2Doc` + `BertSentenceEmbeddings` and put more information into the embedding:

| Annotator | Adds to the chunk | Typical use |
|---|---|---|
| `BertSentenceChunkEmbeddings` | Context: the sentence, or a few tokens around the chunk | Diagnoses and procedures (ICD-10-CM, CPT, SNOMED CT) |
| `EntityChunkEmbeddings` | Related entities: the strength, route and form of a drug | Drugs (RxNorm) |

### **BertSentenceChunkEmbeddings: context-aware codes**

It embeds the chunk and its context, then averages the two. `setStrategy` chooses the context:

- `chunk_only`: the chunk alone, like the pipelines above
- `sentence_average` (default): the chunk and the whole sentence; `setChunkWeight` sets the balance (default 0.5)
- `scope_average`: the chunk and only a few tokens around it, set with `setScopeWindow([before, after])`; the `token` column must also be an input
- `scope_only`: only the tokens around the chunk

<img src="https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/data/Entity%20Resolution%20in%20Spark%20NLP%20for%20Healthcare.jpeg?raw=true" width="800" alt="Entity Resolution in Spark NLP for Healthcare">

In the diagram, *"CT"* alone resolves to an unlisted CT procedure; with its sentence, it resolves to *CT of abdomen*.

In [21]:
context_embeddings = BertSentenceChunkEmbeddings.pretrained("sbiobert_base_cased_mli_onnx", "en", "clinical/models")\
    .setInputCols(["sentence", "token", "ner_chunk"])\
    .setOutputCol("sentence_embeddings")\
    .setCaseSensitive(False)

def resolve_with_context(text, strategy, chunk_weight=0.5, scope_window=(0, 0)):
    context_embeddings.setStrategy(strategy)\
        .setChunkWeight(chunk_weight)\
        .setScopeWindow(list(scope_window))

    context_pipeline = Pipeline(stages=[
        document_assembler,
        sentence_detector,
        tokenizer,
        word_embeddings,
        clinical_ner,
        ner_converter,
        context_embeddings,
        icd10_resolver
    ])

    context_lp = LightPipeline(context_pipeline.fit(empty_data))
    codes = get_codes(context_lp.fullAnnotate(text)[0], "icd10cm_code", top_k=3)
    codes.insert(0, "setting", f"{strategy}, chunk_weight={chunk_weight}, scope_window={scope_window}")
    return codes

sbiobert_base_cased_mli_onnx download started this may take some time.
Approximate size to download 384.4 MB
[OK!]


In [ ]:
examples = [
    "At 32 weeks of pregnancy, she developed hypertension.",
    "There is a benign tumor in his right adrenal shown by CT from his abdomen.",
]

def compare_context(text):
    chunk_only = resolve_with_context(text, "chunk_only")
    with_sentence = resolve_with_context(text, "sentence_average")
    if chunk_only.empty:
        return None  # no PROBLEM chunk in this sentence

    return pd.DataFrame({
        "chunk": chunk_only["chunk"],
        "chunk_only": chunk_only["code"] + ": " + chunk_only["resolved_text"],
        "with_sentence": with_sentence["code"] + ": " + with_sentence["resolved_text"],
    })

pd.concat([compare_context(text) for text in examples], ignore_index=True)

Each row is the same chunk and the same resolver; only the embeddings change.

- **Context helps when the chunk is missing information.** *"hypertension"* alone resolves to essential hypertension (I10). The sentence adds *32 weeks of pregnancy*, and the code moves to gestational hypertension (O13), which matches new high blood pressure during pregnancy.
- **Context can hurt when the chunk is already complete.** *"pregnancy"* alone is a pregnant state (Z33.1); with the sentence, it drifts to a pregnancy complication (O21.0, hyperemesis). *"a benign tumor in his right adrenal"* already names the side, so chunk-only gives the most specific code (D35.01) and the sentence makes it more general (D35.0).

So use context selectively: for example, only for entity types whose chunks tend to be short, or with a small `scope_window` (`scope_average` / `scope_only`) instead of the whole sentence. Always measure the effect on your own notes.

In [23]:
posology_ner = MedicalNerModel.pretrained("ner_posology_large", "en", "clinical/models")\
    .setInputCols(["sentence", "token", "word_embeddings"])\
    .setOutputCol("posology_ner")

posology_converter = NerConverterInternal()\
    .setInputCols(["sentence", "token", "posology_ner"])\
    .setOutputCol("posology_chunk")

pos_tagger = PerceptronModel.pretrained("pos_clinical", "en", "clinical/models")\
    .setInputCols(["sentence", "token"])\
    .setOutputCol("pos_tag")

dependency_parser = DependencyParserModel.pretrained("dependency_conllu", "en")\
    .setInputCols(["sentence", "pos_tag", "token"])\
    .setOutputCol("dependencies")

drug_chunk_embeddings = EntityChunkEmbeddings.pretrained("sbiobert_base_cased_mli", "en", "clinical/models")\
    .setInputCols(["posology_chunk", "dependencies"])\
    .setOutputCol("drug_chunk_embeddings")\
    .setMaxSyntacticDistance(3)\
    .setTargetEntities({"DRUG": ["STRENGTH", "ROUTE", "FORM"]})\
    .setEntityWeights({"DRUG": 0.8, "STRENGTH": 0.2, "ROUTE": 0.2, "FORM": 0.2})

rxnorm_re_resolver = SentenceEntityResolverModel.pretrained("sbiobertresolve_rxnorm_augmented_re", "en", "clinical/models")\
    .setInputCols(["drug_chunk_embeddings"])\
    .setOutputCol("rxnorm_code")\
    .setDistanceFunction("EUCLIDEAN")

drug_pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer,
    word_embeddings,
    posology_ner,
    posology_converter,
    pos_tagger,
    dependency_parser,
    drug_chunk_embeddings,
    rxnorm_re_resolver
])

drug_lp = LightPipeline(drug_pipeline.fit(empty_data))

ner_posology_large download started this may take some time.
Approximate size to download 13.8 MB
[OK!]
pos_clinical download started this may take some time.
Approximate size to download 1.5 MB
[OK!]
dependency_conllu download started this may take some time.
Approximate size to download 16.7 MB
[OK!]
sbiobert_base_cased_mli download started this may take some time.
Approximate size to download 384.3 MB
[OK!]
sbiobertresolve_rxnorm_augmented_re download started this may take some time.
Approximate size to download 724.6 MB
[OK!]


In [24]:
drug_text = "The patient was given metformin 400 mg, coumadin 5 mg, coumadin, amlodipine 10 MG tablet"

drug_result = drug_lp.fullAnnotate(drug_text)[0]

pd.DataFrame({
    "merged_chunk": [embedding.result for embedding in drug_result["drug_chunk_embeddings"]],
    "rxnorm_code": [code.result for code in drug_result["rxnorm_code"]],
    "resolved_text": [code.metadata["resolved_text"] for code in drug_result["rxnorm_code"]],
})

,merged_chunk,rxnorm_code,resolved_text
0,metformin 400 mg,332809,metformin 400 MG
1,coumadin 5 mg,855333,warfarin sodium 5 MG [Coumadin]
2,coumadin,202421,Coumadin
3,amlodipine 10 MG tablet,308135,amlodipine 10 MG Oral Tablet


The strength and form are merged into the drug chunk. In our test run, *coumadin 5 mg* resolved to a specific product (warfarin sodium 5 MG), while the second *coumadin*, with no dose next to it, stayed at brand level: the same limitation we saw in Section 6.

More examples: [24.1.Improved_Entity_Resolution_with_SentenceChunkEmbeddings](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/24.1.Improved_Entity_Resolution_with_SentenceChunkEmbeddings.ipynb) and [3.2.Sentence_Entity_Resolvers_with_EntityChunkEmbeddings](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/3.2.Sentence_Entity_Resolvers_with_EntityChunkEmbeddings.ipynb).

## **8. The Shortcut: Pretrained Resolver Pipelines**

Everything built by hand above also comes packaged as **pretrained pipelines**: NER, resolver and, in some pipelines, mappers. One line to load, one line to run.

In [25]:
icd10_pretrained = PretrainedPipeline("icd10cm_resolver_pipeline", "en", "clinical/models")

text = """A 28-year-old female with a history of gestational diabetes mellitus diagnosed eight years and anisakiasis. Also, it was reported that fetal and neonatal hemorrhage"""

pretrained_result = icd10_pretrained.fullAnnotate(text)[0]

pd.DataFrame({
    "chunk": [chunk.result for chunk in pretrained_result["icd10cm_ner_chunk"]],
    "entity": [chunk.metadata["entity"] for chunk in pretrained_result["icd10cm_ner_chunk"]],
    "icd10cm_code": [code.result for code in pretrained_result["icd10cm"]],
})

icd10cm_resolver_pipeline download started this may take some time.
Approx size to download 2.4 GB
[OK!]


,chunk,entity,icd10cm_code
0,gestational diabetes mellitus,PROBLEM,O24.4
1,anisakiasis,PROBLEM,B81.0
2,fetal and neonatal hemorrhage,PROBLEM,P54.5


**More resolver pipelines**

| Pipeline | Entities | Terminology |
|---|---|---|
| [`rxnorm_resolver_pipeline`](https://nlp.johnsnowlabs.com/2024/09/17/rxnorm_resolver_pipeline_en.html) | Drug | RxNorm |
| [`snomed_resolver_pipeline`](https://nlp.johnsnowlabs.com/2024/03/11/snomed_resolver_pipeline_en.html) | Clinical findings | SNOMED CT |
| [`loinc_resolver_pipeline`](https://nlp.johnsnowlabs.com/2024/10/08/loinc_resolver_pipeline_en.html) | Test | LOINC |
| [`hcc_resolver_pipeline`](https://nlp.johnsnowlabs.com/2024/01/17/hcc_resolver_pipeline_en.html) | Clinical findings | HCC |
| [`icd10cm_rxnorm_resolver_pipeline`](https://nlp.johnsnowlabs.com/2024/09/18/icd10cm_rxnorm_resolver_pipeline_en.html) | Problem, Drug | ICD-10-CM, RxNorm |
| [`medication_resolver_pipeline`](https://nlp.johnsnowlabs.com/2024/12/19/medication_resolver_pipeline_en.html) | Drug | RxNorm, NDC, UMLS, SNOMED CT, adverse reactions |
| [`umls_major_concepts_resolver_pipeline`](https://nlp.johnsnowlabs.com/2024/12/24/umls_major_concepts_resolver_pipeline_en.html) | Major clinical concepts | UMLS CUI |
| [`hpo_resolver_pipeline`](https://nlp.johnsnowlabs.com/2024/01/18/hpo_resolver_pipeline_en.html) | Human phenotype | HPO |

The full list of resolver pipelines is in [3.Clinical_Entity_Resolvers](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/3.Clinical_Entity_Resolvers.ipynb).

## **9. Takeaways**

- NER finds the mentions; **resolvers map them to standard codes**.
- A resolver is **sentence embeddings + nearest-neighbour search** over a terminology. Use the embeddings model from the model card.
- **Resolve the right chunks:** whitelist the NER labels that fit the terminology, and drop negated, hypothetical and family-history mentions with `AssertionFilterer`.
- **Chunk quality = code quality:** the resolver only sees the chunk. When the chunk isn't enough, add context with `BertSentenceChunkEmbeddings` or the drug's strength and form with `EntityChunkEmbeddings`.
- Keep the **top-k candidates**: they're the basis for human review and code refinement.
- `Router` runs several resolvers on a single embeddings pass; `PipelineOutputParser` turns the output into structured JSON.
- Start with a **pretrained pipeline**; build your own when you need control.

### **Going further**

- [3.Clinical_Entity_Resolvers](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/3.Clinical_Entity_Resolvers.ipynb): the full reference, with CPT, LOINC, UMLS, HPO, `ResolverMerger`, `MultiChunk2Doc` and a table of recommended NER models and labels for each resolver
- [3.4.Resolving_Medical_Terms_to_Terminology_Codes_Directly](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/3.4.Resolving_Medical_Terms_to_Terminology_Codes_Directly.ipynb)
- [13.Snomed_Entity_Resolver_Model_Training](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/13.Snomed_Entity_Resolver_Model_Training.ipynb): train your own resolver
- [13.1.Finetuning_Sentence_Entity_Resolver_Model](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/13.1.Finetuning_Sentence_Entity_Resolver_Model.ipynb): fine-tune an existing resolver
- [11.1.Healthcare_Code_Mapping](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/11.1.Healthcare_Code_Mapping.ipynb): map codes between terminologies (e.g. ICD-10-CM → SNOMED CT)
- [High-accuracy clinical term mapping to standard medical terminologies](https://medium.com/john-snow-labs/high-accuracy-clinical-term-mapping-to-standard-medical-terminologies-icd-10-rxnorm-snomed-and-3a0333c95b4c): John Snow Labs blog post on resolver accuracy, including sentence-chunk and entity-chunk embeddings
- [Models Hub: Entity Resolution](https://nlp.johnsnowlabs.com/models?task=Entity+Resolution&edition=Spark+NLP+for+Healthcare)